In [0]:
from pyspark.sql import functions as F

# ---------- Customers ----------
customers = (spark.table("workspace.bronze.customers")
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("signup_date", F.to_date("signup_date"))
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
    .drop("_source_file"))
customers.write.mode("overwrite").saveAsTable("workspace.silver.customers")

# ---------- Products ----------
products = (spark.table("workspace.bronze.products")
    .withColumn("product_id", F.col("product_id").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .dropDuplicates(["product_id"])
    .drop("_source_file"))
products.write.mode("overwrite").saveAsTable("workspace.silver.products")

# ---------- Orders ----------
from pyspark.sql.window import Window

w = Window.partitionBy("order_id").orderBy(F.col("_ingested_at").desc())

clean = (spark.table("workspace.bronze.orders")
    .withColumn("order_id", F.col("order_id").cast("int"))
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("product_id", F.col("product_id").cast("int"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("status", F.lower(F.trim("status")))
    .drop("_source_file", "_rescued_data")
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1").drop("rn"))

is_valid = (F.col("quantity") > 0) & F.col("order_ts").isNotNull()

clean.filter(is_valid).createOrReplaceTempView("orders_updates")
clean.filter(~is_valid).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.orders_quarantine")

spark.sql("CREATE TABLE IF NOT EXISTS workspace.silver.orders AS SELECT * FROM orders_updates WHERE 1=0")

spark.sql("""
  MERGE INTO workspace.silver.orders t
  USING orders_updates s ON t.order_id = s.order_id
  WHEN MATCHED THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *
""")

print("silver.orders rows:", spark.table("workspace.silver.orders").count())
print("quarantined:", spark.table("workspace.silver.orders_quarantine").count())

In [0]:
%sql
SELECT status, COUNT(*) FROM workspace.silver.orders GROUP BY status;   -- should be lowercase only
SELECT COUNT(*) - COUNT(DISTINCT order_id) AS dupes FROM workspace.silver.orders;  -- should be 0